# Sequitur Algorithm

Sequitur (Nevill-Manning & Witten, 1997) is an online algorithm that infers a **context-free grammar** from a flat sequence of symbols in **linear time and space**. It discovers hierarchical, repeated structure by maintaining just two invariants:

1. **Digram uniqueness** — no pair of adjacent symbols appears more than once across all rules.
2. **Rule utility** — every rule (except the start rule) is referenced at least twice.

These two constraints work in tension: digram uniqueness *creates* rules, rule utility *removes* them, and together they keep the grammar minimal.

### References

- Nevill-Manning, C.G. and Witten, I.H. (1997). **"Identifying Hierarchical Structure in Sequences: A linear-time algorithm."** *Journal of Artificial Intelligence Research*, 7, 67–82. [https://doi.org/10.1613/jair.374](https://doi.org/10.1613/jair.374)
- [Wikipedia: Sequitur algorithm](https://en.wikipedia.org/wiki/Sequitur_algorithm)

## Implementation

Each rule's right-hand side is a doubly-linked list of symbols, giving O(1) splicing.
A hash table indexes every digram in the grammar for O(1) duplicate detection.

In [ ]:
class Symbol:
    """A node in a rule's doubly-linked list. Holds either a terminal (str) or a rule reference."""

    def __init__(self, value):
        self.value = value          # str for terminal, Rule for non-terminal
        self.prev = None
        self.next = None

    @property
    def is_guard(self):
        return False

    def key(self):
        return self.value if isinstance(self.value, str) else id(self.value)

    def __repr__(self):
        if isinstance(self.value, str):
            return self.value
        return str(self.value)


class Guard(Symbol):
    """Sentinel that bookends a rule's linked list, pointing back to the rule."""

    def __init__(self, rule):
        super().__init__(rule)
        self.prev = self
        self.next = self

    @property
    def is_guard(self):
        return True

: 

In [ ]:
class Rule:
    """A grammar rule.  The right-hand side is a doubly-linked ring of Symbols
    between a Guard node that also serves as the head pointer."""

    _counter = 0  # class-level counter for readable names

    def __init__(self):
        self.guard = Guard(self)
        self.ref_count = 0
        self.index = Rule._counter
        Rule._counter += 1

    # -- linked-list helpers ---------------------------------------------------

    def first(self):
        return self.guard.next

    def last(self):
        return self.guard.prev

    def append(self, value):
        """Insert a new Symbol at the end of this rule (before the guard)."""
        sym = Symbol(value)
        _insert_after(self.last(), sym)
        return sym

    def symbols(self):
        """Iterate over the symbols in this rule (skipping guards)."""
        s = self.guard.next
        while not s.is_guard:
            yield s
            s = s.next

    def __len__(self):
        return sum(1 for _ in self.symbols())

    def __repr__(self):
        name = 'S' if self.index == 0 else chr(ord('A') + (self.index - 1) % 26)
        return name

    def format(self):
        return f"{self} -> {' '.join(repr(s) for s in self.symbols())}"


def _insert_after(left, sym):
    """Splice *sym* into the linked list right after *left*."""
    right = left.next
    left.next = sym
    sym.prev = left
    sym.next = right
    right.prev = sym


def _remove(sym):
    """Unlink *sym* from its list."""
    sym.prev.next = sym.next
    sym.next.prev = sym.prev

In [ ]:
def _digram_key(a, b):
    return (a.key(), b.key())


class Sequitur:
    """Builds a grammar from a stream of characters."""

    def __init__(self):
        Rule._counter = 0
        self.start = Rule()          # the S rule
        self.digrams = {}            # (key, key) -> Symbol (first of the pair)

    # -- public API ------------------------------------------------------------

    def feed(self, text):
        """Feed a string one character at a time."""
        for ch in text:
            self._append(ch)

    def rules(self):
        """Yield all rules reachable from S (breadth-first)."""
        seen = set()
        queue = [self.start]
        while queue:
            rule = queue.pop(0)
            if id(rule) in seen:
                continue
            seen.add(id(rule))
            yield rule
            for sym in rule.symbols():
                if isinstance(sym.value, Rule):
                    queue.append(sym.value)

    def expand(self, rule=None):
        """Recursively expand a rule back into the original string."""
        if rule is None:
            rule = self.start
        parts = []
        for sym in rule.symbols():
            if isinstance(sym.value, str):
                parts.append(sym.value)
            else:
                parts.append(self.expand(sym.value))
        return ''.join(parts)

    # -- internals -------------------------------------------------------------

    def _append(self, ch):
        new_sym = self.start.append(ch)
        self._check_digram(new_sym.prev)

    def _check_digram(self, sym):
        """Enforce digram uniqueness starting at *sym* (the left symbol of the pair)."""
        if sym.is_guard or sym.next.is_guard:
            return

        key = _digram_key(sym, sym.next)
        existing = self.digrams.get(key)

        if existing is None:
            self.digrams[key] = sym
            return

        if existing is sym:
            return

        # Duplicate digram found — either reuse an existing rule or create one.
        if (existing.prev.is_guard and existing.next.next.is_guard
                and isinstance(existing.prev.value, Rule)):
            # The existing digram *is* an entire rule body -> reuse that rule.
            rule = existing.prev.value
            self._replace_digram(sym, rule)
        else:
            # Create a new rule for this repeated digram.
            rule = Rule()
            rule.append(existing.value)
            rule.append(existing.next.value)
            self._replace_digram(existing, rule)
            self._replace_digram(sym, rule)
            self.digrams[_digram_key(rule.first(), rule.first().next)] = rule.first()

    def _replace_digram(self, sym, rule):
        """Replace the digram starting at *sym* with a reference to *rule*."""
        right = sym.next

        # Remove the old digrams touching these symbols.
        self._unregister(sym)
        if not right.next.is_guard:
            self._unregister(right)

        # Decrement ref counts for any non-terminal being removed.
        if isinstance(sym.value, Rule):
            sym.value.ref_count -= 1
            self._enforce_utility(sym.value)
        if isinstance(right.value, Rule):
            right.value.ref_count -= 1
            self._enforce_utility(right.value)

        # Splice out the two symbols, insert the rule reference.
        _remove(right)
        sym.value = rule
        rule.ref_count += 1

        # Re-check the new digrams formed on each side.
        if not sym.prev.is_guard:
            self._check_digram(sym.prev)
        if not sym.next.is_guard:
            self._check_digram(sym)

    def _enforce_utility(self, rule):
        """If a rule is referenced only once, inline it and delete."""
        if rule.ref_count > 1:
            return

        # Find the single reference to this rule in the grammar.
        ref = self._find_sole_reference(rule)
        if ref is None:
            return

        # Splice the rule body in place of the reference.
        first = rule.first()
        last = rule.last()

        self._unregister(ref)
        if not ref.prev.is_guard:
            self._unregister(ref.prev)

        # Re-link: ref.prev <-> first ... last <-> ref.next
        ref.prev.next = first
        first.prev = ref.prev
        ref.next.prev = last
        last.next = ref.next

        # Bump ref counts for the inlined symbols.
        for s in rule.symbols():
            if isinstance(s.value, Rule):
                # They were already counted under the old rule, no net change.
                pass

        # Register new digrams at the join points.
        if not first.prev.is_guard:
            self._check_digram(first.prev)
        if not last.next.is_guard:
            self._check_digram(last)

    def _find_sole_reference(self, rule):
        """Find the single Symbol that references *rule* across the whole grammar."""
        for r in self.rules():
            for sym in r.symbols():
                if sym.value is rule:
                    return sym
        return None

    def _unregister(self, sym):
        if sym.is_guard or sym.next.is_guard:
            return
        key = _digram_key(sym, sym.next)
        if self.digrams.get(key) is sym:
            del self.digrams[key]

    def __repr__(self):
        return '\n'.join(r.format() for r in self.rules())

## Example: Step by Step

Let's trace the algorithm on a small string to see rules form and collapse.

In [ ]:
seq = Sequitur()
text = "abcabcabc"

for i, ch in enumerate(text, 1):
    seq.feed(ch)
    print(f"After reading '{text[:i]}':")
    print(f"  {seq}")
    print()

In [ ]:
# Verify the grammar expands back to the original string
assert seq.expand() == text
print(f"Original : {text}")
print(f"Expanded : {seq.expand()}")
print(f"\nFinal grammar:")
print(seq)

## Longer Example

A longer, more repetitive string shows deeper hierarchical structure.

In [ ]:
seq2 = Sequitur()
longer = "abcdabcdabcdabcd"
seq2.feed(longer)

print(f"Input ({len(longer)} chars): {longer}")
print(f"\nGrammar:")
print(seq2)
print(f"\nExpands to: {seq2.expand()}")
assert seq2.expand() == longer

In [ ]:
# Counting the total symbols in the grammar vs the original length
grammar_symbols = sum(len(r) for r in seq2.rules())
print(f"Original length   : {len(longer)}")
print(f"Grammar symbols   : {grammar_symbols}")
print(f"Compression ratio : {grammar_symbols / len(longer):.2f}")

## How It Works — Recap

1. **Scan left to right.** Each new character is appended to the start rule S.
2. **Check the new digram** (the last two symbols of S) against a hash table.
   - If unseen, register it.
   - If already seen, either **reuse** the existing rule whose body matches, or **create** a new rule and replace both occurrences.
3. **Enforce rule utility.** If any rule's reference count drops to 1, inline it and delete.
4. Repeat until the input is consumed.

The result is a grammar that generates exactly the input string, with repeated substrings factored into reusable rules — all in **O(n) time and space**.